# Notebook 03 — Bilinear Image Interpolation & Debugging Degenerate Cases

**Session:** Interpolation Practice Lab  
**Module:** 02 · Calculus and Linear Algebra

## Goal

1. Understand and reproduce the **integer-coordinate zeroing bug** in a naive bilinear resizer (`bl_resize_normalcase`).
2. Fix the bug using the **four-branch case handler** (`bl_resize_special`) for grayscale images (`../data/tree.jpg`).
3. Extend bilinear resizing to **3-channel RGB/BGR images** (`bl_resize_color`).
4. Compare the pure-Python loop implementation against a **vectorized NumPy** version and OpenCV's `cv2.INTER_LINEAR`.


In [ ]:
import math
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

IMAGE_PATH = Path("../data/tree.jpg")
OUTPUT_DIR = Path("../outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. The Naive Bilinear Implementation & Why It Fails (`bl_resize_normalcase`)

When `x` is an integer, `x_floor == x_ceil == x`, making `(x_ceil - x) == 0` and `(x - x_floor) == 0`, so `q1 = 0`, `q2 = 0`, and `q = 0`.  
When `y` is an integer, `y_floor == y_ceil == y`, making `(y_ceil - y) == 0` and `(y - y_floor) == 0`, so `q = 0` even if `q1` and `q2` are nonzero.  
The result is a dark grid across 75% of a $2\times$-upsampled image.


In [ ]:
def bl_resize_normalcase(
    original_img: list[list[int]],
    old_h: int,
    old_w: int,
    new_h: int,
    new_w: int,
) -> np.ndarray:
    """Naive bilinear resize that produces 0s whenever x or y is an integer."""
    resized = np.zeros((new_h, new_w), dtype=np.float64)
    w_scale_factor = old_w / new_w if new_w != 0 else 0.0
    h_scale_factor = old_h / new_h if new_h != 0 else 0.0

    for i in range(new_h):
        for j in range(new_w):
            x = i * h_scale_factor
            y = j * w_scale_factor
            x_floor = math.floor(x)
            x_ceil = min(old_h - 1, math.ceil(x))
            y_floor = math.floor(y)
            y_ceil = min(old_w - 1, math.ceil(y))

            v1 = original_img[x_floor][y_floor]
            v2 = original_img[x_ceil][y_floor]
            v3 = original_img[x_floor][y_ceil]
            v4 = original_img[x_ceil][y_ceil]

            q1 = v1 * (x_ceil - x) + v2 * (x - x_floor)
            q2 = v3 * (x_ceil - x) + v4 * (x - x_floor)
            q = q1 * (y_ceil - y) + q2 * (y - y_floor)
            resized[i, j] = q

    return resized.astype(np.uint8)

## 2. Boundary-Aware Four-Branch Bilinear Resizing (`bl_resize_special`)

Checking `x_ceil == x_floor` and `y_ceil == y_floor` handles all four coordinate geometries:
1. **Exact grid intersection** (`x_ceil == x_floor` and `y_ceil == y_floor`) -> copy `original_img[int(x)][int(y)]`.
2. **Horizontal grid line / bottom border** (`x_ceil == x_floor`) -> 1D linear interpolation along `y`.
3. **Vertical grid line / right border** (`y_ceil == y_floor`) -> 1D linear interpolation along `x`.
4. **Interior cell** -> full four-corner bilinear interpolation.


In [ ]:
def bl_resize_special(
    original_img: list[list[int]],
    old_h: int,
    old_w: int,
    new_h: int,
    new_w: int,
) -> np.ndarray:
    """Bilinear resize for grayscale images handling integer coordinates and borders."""
    resized = np.zeros((new_h, new_w), dtype=np.float64)
    w_scale_factor = old_w / new_w if new_w != 0 else 0.0
    h_scale_factor = old_h / new_h if new_h != 0 else 0.0

    for i in range(new_h):
        for j in range(new_w):
            x = i * h_scale_factor
            y = j * w_scale_factor
            x_floor = math.floor(x)
            x_ceil = min(old_h - 1, math.ceil(x))
            y_floor = math.floor(y)
            y_ceil = min(old_w - 1, math.ceil(y))

            if (x_ceil == x_floor) and (y_ceil == y_floor):
                q = original_img[int(x)][int(y)]
            elif x_ceil == x_floor:
                q1 = original_img[int(x)][int(y_floor)]
                q2 = original_img[int(x)][int(y_ceil)]
                q = q1 * (y_ceil - y) + q2 * (y - y_floor)
            elif y_ceil == y_floor:
                q1 = original_img[int(x_floor)][int(y)]
                q2 = original_img[int(x_ceil)][int(y)]
                q = q1 * (x_ceil - x) + q2 * (x - x_floor)
            else:
                v1 = original_img[x_floor][y_floor]
                v2 = original_img[x_ceil][y_floor]
                v3 = original_img[x_floor][y_ceil]
                v4 = original_img[x_ceil][y_ceil]

                q1 = v1 * (x_ceil - x) + v2 * (x - x_floor)
                q2 = v3 * (x_ceil - x) + v4 * (x - x_floor)
                q = q1 * (y_ceil - y) + q2 * (y - y_floor)

            resized[i, j] = q

    return np.clip(resized, 0, 255).astype(np.uint8)


img_gray = cv2.imread(str(IMAGE_PATH), cv2.IMREAD_GRAYSCALE)
assert img_gray is not None, f"Could not load {IMAGE_PATH}"
h, w = img_gray.shape
img_list = img_gray.tolist()

buggy_2x = bl_resize_normalcase(img_list, h, w, h * 2, w * 2)
fixed_2x = bl_resize_special(img_list, h, w, h * 2, w * 2)

# Verify fraction of zeros in buggy vs fixed output
zero_ratio_buggy = float(np.mean(buggy_2x == 0))
zero_ratio_fixed = float(np.mean(fixed_2x == 0))
print(f"Buggy zero-pixel ratio: {zero_ratio_buggy:.3f}")
print(f"Fixed zero-pixel ratio: {zero_ratio_fixed:.3f}")
assert zero_ratio_buggy > 0.70

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(buggy_2x[150:250, 150:250], cmap="gray")
axes[0].set_title("Naive Buggy (Zoomed Grid Artifact)")
axes[0].axis("off")

axes[1].imshow(fixed_2x, cmap="gray")
axes[1].set_title(f"Fixed 4-Branch Bilinear {fixed_2x.shape}")
axes[1].axis("off")

cv_bilinear = cv2.resize(img_gray, (w * 2, h * 2), interpolation=cv2.INTER_LINEAR)
axes[2].imshow(cv_bilinear, cmap="gray")
axes[2].set_title(f"OpenCV INTER_LINEAR {cv_bilinear.shape}")
axes[2].axis("off")

plt.tight_layout()
plt.show()

## 3. Multi-Channel Color Bilinear Resizing (`bl_resize_color`) & Vectorized NumPy


In [ ]:
def bl_resize_color(original_img: np.ndarray, new_h: int, new_w: int) -> np.ndarray:
    """Bilinear resize for 3-channel (H, W, C) images."""
    old_h, old_w, c = original_img.shape
    resized = np.zeros((new_h, new_w, c), dtype=np.float64)
    w_scale_factor = old_w / new_w if new_w != 0 else 0.0
    h_scale_factor = old_h / new_h if new_h != 0 else 0.0

    for i in range(new_h):
        for j in range(new_w):
            x = i * h_scale_factor
            y = j * w_scale_factor
            x_floor = math.floor(x)
            x_ceil = min(old_h - 1, math.ceil(x))
            y_floor = math.floor(y)
            y_ceil = min(old_w - 1, math.ceil(y))

            if (x_ceil == x_floor) and (y_ceil == y_floor):
                q = original_img[int(x), int(y), :]
            elif x_ceil == x_floor:
                q1 = original_img[int(x), int(y_floor), :]
                q2 = original_img[int(x), int(y_ceil), :]
                q = q1 * (y_ceil - y) + q2 * (y - y_floor)
            elif y_ceil == y_floor:
                q1 = original_img[int(x_floor), int(y), :]
                q2 = original_img[int(x_ceil), int(y), :]
                q = q1 * (x_ceil - x) + q2 * (x - x_floor)
            else:
                v1 = original_img[x_floor, y_floor, :]
                v2 = original_img[x_ceil, y_floor, :]
                v3 = original_img[x_floor, y_ceil, :]
                v4 = original_img[x_ceil, y_ceil, :]

                q1 = v1 * (x_ceil - x) + v2 * (x - x_floor)
                q2 = v3 * (x_ceil - x) + v4 * (x - x_floor)
                q = q1 * (y_ceil - y) + q2 * (y - y_floor)

            resized[i, j, :] = q

    return np.clip(resized, 0, 255).astype(np.uint8)


def bl_resize_vectorized(original_img: np.ndarray, new_h: int, new_w: int) -> np.ndarray:
    """Fast vectorized bilinear resize matching bl_resize_color coordinate mapping."""
    old_h, old_w = original_img.shape[:2]
    x = np.arange(new_h, dtype=np.float64) * (old_h / new_h)
    y = np.arange(new_w, dtype=np.float64) * (old_w / new_w)
    gx, gy = np.meshgrid(x, y, indexing="ij")

    x0 = np.floor(gx).astype(np.int32)
    y0 = np.floor(gy).astype(np.int32)
    x1 = np.minimum(x0 + 1, old_h - 1)
    y1 = np.minimum(y0 + 1, old_w - 1)

    wx = gx - x0
    wy = gy - y0

    if original_img.ndim == 3:
        wx = wx[..., None]
        wy = wy[..., None]

    v1 = original_img[x0, y0].astype(np.float64)
    v2 = original_img[x1, y0].astype(np.float64)
    v3 = original_img[x0, y1].astype(np.float64)
    v4 = original_img[x1, y1].astype(np.float64)

    q = v1 * (1.0 - wx) * (1.0 - wy) + v2 * wx * (1.0 - wy) + v3 * (1.0 - wx) * wy + v4 * wx * wy
    return np.clip(q, 0, 255).astype(np.uint8)


# Test on a synthetic RGB gradient patch and on tree.jpg loaded as RGB
img_bgr = cv2.imread(str(IMAGE_PATH), cv2.IMREAD_COLOR)
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
color_loop = bl_resize_color(img_rgb[:100, :100, :], 200, 200)
color_vec = bl_resize_vectorized(img_rgb[:100, :100, :], 200, 200)
assert np.array_equal(color_loop, color_vec)
print("Color loop and vectorized bilinear outputs match identically!")